## **1.Setting**

In [1]:
%pip install -q \
    "vllm==0.11.1" \
    "numpy==2.2.6" \
    "pandas>=2.2,<3.0" \
    "datasets>=3.6,<4.0" \
    "huggingface_hub>=0.30,<1.0" \
    "transformers>=4.50,<4.58" \
    "peft>=0.15,<0.18" \
    "openai>=2.0,<3.0" \
    "tqdm>=4.66,<5.0"

Note: you may need to restart the kernel to use updated packages.


In [3]:
import re
import json
import pandas as pd
from typing import List, Dict
from datasets import load_dataset, Dataset
from vllm import LLM, SamplingParams
from huggingface_hub import login

import os
# Hugging Face 로그인은 다음 데이터셋 로딩 셀보다 먼저 수행됩니다
# Hugging Face 로그인 (새 토큰은 HF_TOKEN 환경변수 또는 실행 중 보안 입력창에서 받음)
# 주의: Hugging Face 토큰을 노트북 코드에 직접 붙여넣거나 출력하지 마세요
from getpass import getpass
from huggingface_hub import login, HfApi
HF_TOKEN = (os.getenv("HF_TOKEN") or "").strip()
if not HF_TOKEN:
    HF_TOKEN = getpass("새 Hugging Face access token 입력 (입력값 숨김): ").strip()
if not HF_TOKEN:
    raise ValueError("HF_TOKEN이 필요합니다. 새 토큰을 입력하거나 환경변수 HF_TOKEN을 설정하세요.")
login(token=HF_TOKEN)
HF_USERNAME = (os.getenv("PILATES_HF_USERNAME") or HfApi().whoami(token=HF_TOKEN)["name"]).strip()
print("Hugging Face 로그인 확인:", HF_USERNAME)  # 토큰은 출력하지 않음
DATASET_ID = os.getenv("PILATES_DATASET_REPO", f"{HF_USERNAME}/pilates-function-calling-datasets-korean-v2")
FINETUNED_MODEL = os.getenv("PILATES_MODEL_REPO", f"{HF_USERNAME}/Qwen2.5-7B-Instruct-pilates-function-calling-v2")
from transformers import AutoTokenizer


새 Hugging Face access token 입력 (입력값 숨김):  ········


Hugging Face 로그인 확인: somsom317


## **2.Test 데이터 전처리**

In [4]:
# 위 Setting 셀에서 검증된 Hugging Face 로그인 사용
dataset = load_dataset(DATASET_ID)
assert "train" in dataset and "test" in dataset, "Part1에서 V2 train/test 분리 후 업로드해야 합니다."


test_ratio = 0.2
total_len = len(dataset["train"]) + len(dataset["test"])
test_size = len(dataset["test"])


test_indices = list(range(len(dataset["test"])))
train_indices = list(range(len(dataset["train"])))


def format_conversations(sample):
    return {
        "messages": [
            {"role": "system", "content": sample["system_prompt"]},
            *sample["messages"]
        ]
    }
    

train_dataset = [format_conversations(dataset["train"][i]) for i in train_indices]
test_dataset = [format_conversations(dataset["test"][i]) for i in test_indices]


train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)


print(f"\n전체 데이터 분할 결과: Train {len(train_dataset)}개, Test {len(test_dataset)}개")

print(train_dataset[min(5,len(train_dataset)-1)]["messages"])


README.md: 0.00B [00:00, ?B/s]


전체 데이터 분할 결과: Train 64개, Test 16개
[{'content': '당신은 글로리 필라테스의 AI 상담사입니다. 정확하고 자연스럽게 상담하십시오.\n\n행동 선택 규칙:\n- 이미 알고 있는 정보로 답할 수 있으면 바로 답하고, 실제 센터/회원 DB 정보가 필요하면 Tool을 사용합니다.\n- 선택 인자가 누락되어도 전체 조회가 가능하면 강사·요일 등을 불필요하게 되묻지 않고 바로 Tool을 호출합니다.\n- 모호한 지시 대상이나 서로 충돌하는 조건처럼 Tool·대화 맥락으로도 해소할 수 없는 필수 정보가 있을 때만 한 가지를 되묻습니다.\n- 사용자가 그 모호함을 풀어주면 다시 물어보지 말고 곧바로 필요한 작업을 처리합니다.\n- 월요일·수요일처럼 양립 가능한 복수 조건은 필요 시 여러 Tool을 호출합니다.\n- 이전 Tool 결과만으로 답할 수 있으면 불필요하게 재조회하지 않습니다.\n- 예약 생성·취소·결제 등 Tool에 없는 작업은 실행했다고 말하지 않습니다.\n- 함수 호출 전 별도의 자연어 안내 멘트를 만들지 않습니다.\n\n로그인한 사용자의 현재 ID: U001\n오늘 날짜: 2026-11-02\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>\n{"type": "function", "function": {"name": "search_policy_info", "description": "센터의 예약 오픈, 취소, 대기, 출석, 환불 및 운영시간에 관한 정책과 약관을 조회합니다. \'대기 예약 가능해?\', \'취소하면 횟수 돌려받아?\'처럼 규정을 묻는 질문에 사용합니다. 실제 빈자리나 회원 본인의 예약 내역을 조회하는 함수는 아닙니다.", "parameters": {"type": 

In [5]:
# 학습과 정확히 같은 Qwen Chat Template: 역할·공백·토큰 경계 일치
# Base/FT 모두 동일 test split, 동일 입력, 동일 디코딩 옵션
tokenizer=AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
def to_chatml(messages):
    return tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=False)

def extract_examples(data):
    messages=data["messages"] if isinstance(data,dict) else data
    examples=[]
    for idx,msg in enumerate(messages):
        if msg["role"]!="assistant":continue
        previous_messages=messages[:idx]
        input_text=tokenizer.apply_chat_template(
            previous_messages,tokenize=False,add_generation_prompt=True
        )
        examples.append({"input":input_text,"label":msg["content"].strip()})
    return examples
prompt_lst=[];label_lst=[]
for item in test_dataset:
    for ex in extract_examples(item):
        prompt_lst.append(ex["input"]);label_lst.append(ex["label"])
print(f"원본 테스트 대화 수: {len(test_dataset)}")
print(f"Assistant 응답별 테스트 샘플 수: {len(prompt_lst)}")
print(f"정답 Tool Call: {sum('<tool_call>' in x for x in label_lst)}")
print(f"정답 비호출(되묻기/일상/지원불가/조회결과 답변): {sum('<tool_call>' not in x for x in label_lst)}")
if prompt_lst:
    print("\n[Input 예시]", prompt_lst[0][-1300:])
    print("\n[Label 예시]", label_lst[0])


원본 테스트 대화 수: 16
Assistant 응답별 테스트 샘플 수: 130
정답 Tool Call: 48
정답 비호출(되묻기/일상/지원불가/조회결과 답변): 82

[Input 예시] ": "string", "description": "HH:MM 형식입니다. between에서는 수업 시작 시각 검색 범위의 상한(양 끝 포함), 그 외에는 수업 자체 종료 시각의 상한(이하)입니다."}, "time_filter": {"type": "string", "enum": ["exact", "after", "at_or_after", "before", "at_or_before", "between"], "description": "수업의 시작 시각 기준: exact=정각 시작(==), before='이전' 미만(<), at_or_before='까지' 이하(<=), after='초과' 기준 시각 제외(>), at_or_after='부터' 이상(>=), between='A부터 B까지' 시작 시각 구간 양끝 포함입니다. 예: 3시 이전은 15:00 제외, 3시부터는 15:00 포함. 생략하면 exact입니다."}}, "required": [], "additionalProperties": false}}}
{"type": "function", "function": {"name": "get_my_memberships", "description": "로그인한 회원 본인의 보유 이용권, 상품 유형, 유효기간, 남은 횟수를 조회합니다. 단일 및 콤보 이용권을 모두 반환하며 콤보 상품은 remaining_sessions_by_type에서 '7:1 레슨'(그룹), '2:1 듀엣 레슨'(듀엣), '1:1 개인 레슨'(개인)의 남은 횟수를 각각 확인합니다. 상품 구매 가능 목록은 get_membership_info, 실제 빈자리는 get_available_classes, 이미 예약한 날짜는 get_my_bookings로 확인합니다. 로그인 회원은 내부에서 결정하며 user_id 인자는 받지 않습니

1. tool_selection : 정답 함수와 예측 함수가 정확히 일치하는지

2. params_selection: 정답 키를 포함했는지 + 불필요한 키를 생성했는지

3. params_value_accurac : 파라미터의 내부 값이 모두 일치하는지

## **3.Fine Tuned Model 호출**

In [6]:
sampling_params = SamplingParams(temperature=0,max_tokens=2048,stop=["<|im_end|>"])
llm = LLM(model=FINETUNED_MODEL)
fine_tuned_outputs = llm.generate(prompt_lst, sampling_params=sampling_params)
fine_tuned_text_results = [sample.outputs[0].text.strip() for sample in fine_tuned_outputs]
print(fine_tuned_text_results[20])


df = pd.DataFrame({"prompt": prompt_lst,"label": label_lst,"output": fine_tuned_text_results})


df.to_csv("evaluation_results.csv", index=False, encoding="utf-8-sig")
print("\n평가 결과가 'evaluation_results.csv' 파일로 저장되었습니다.")
df.head(10)


for label, pred in zip(df['label'].to_list()[:50], df['output'].to_list()[:50]):
    print('레이블 :', label)
    print('--' * 50)
    print('모델의 예측:', pred)
    print('==' * 50)


INFO 11-12 08:23:32 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'somsom317/Qwen2.5-7B-Instruct-pilates-function-calling-v2'}


config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

INFO 11-12 08:23:45 [model.py:631] Resolved architecture: Qwen2ForCausalLM
INFO 11-12 08:23:45 [model.py:1745] Using max model len 32768
INFO 11-12 08:23:46 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (14 outputs left out)

(EngineCore_DP0 pid=5271) INFO 11-12 08:25:37 [gpu_model_runner.py:4240] Graph capturing finished in 5 secs, took 0.49 GiB
(EngineCore_DP0 pid=5271) INFO 11-12 08:25:37 [core.py:250] init engine (profile, create kv cache, warmup model) took 25.71 seconds
INFO 11-12 08:25:39 [llm.py:352] Supported tasks: ['generate']


Adding requests:   0%|          | 0/130 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/130 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

월요일에는 오전 10시 박지훈 강사님 수업(CL101, 잔여 1석), 오후 2시 이수진 강사님 수업(CL102, 잔여 1석)이 있습니다. 수요일에는 오후 4시 김민지 강사님 수업(CL103, 잔여 1석)이 있습니다. 추가로 궁금한 점이 있으신가요?

평가 결과가 'evaluation_results.csv' 파일로 저장되었습니다.
레이블 : 안녕하세요! 글로리 필라테스 AI 상담사입니다. 어떤 요일이나 수업 유형을 특정해서 조회해 드릴까요?
----------------------------------------------------------------------------------------------------
모델의 예측: <tool_call>
{"name": "get_class_schedule", "arguments": {}}
</tool_call>
레이블 : <tool_call>
{"name": "get_class_schedule", "arguments": {"day_of_week": "월요일"}}
</tool_call>
<tool_call>
{"name": "get_class_schedule", "arguments": {"day_of_week": "수요일"}}
</tool_call>
----------------------------------------------------------------------------------------------------
모델의 예측: <tool_call>
{"name": "get_class_schedule", "arguments": {"day_of_week": "월요일"}}
</tool_call>
<tool_call>
{"name": "get_class_schedule", "arguments": {"day_of_week": "수요일"}}
</tool_call>
레이블 : 월요일 수업:  
- 09:00~09:50 김민지 강사님의 7:1 레슨  
- 11:00~11:50 박지훈 강사님의 2:1 듀엣 레슨  


In [7]:
# 기존 지표 4개 유지 + 호출 판단/다중 호출/비호출 오류를 함께 계산
# IMPORTANT: 자연어 CLARIFY의 의미적 적절성은 별도의 수작업 평가가 필요합니다
def extract_tool_calls(text):
    matches=re.findall(r'<tool_call>\s*(.*?)\s*</tool_call>',str(text),re.S)
    calls=[]
    for raw in matches:
        try:
            val=json.loads(raw)
        except (ValueError,TypeError):
            return None   # malformed JSON
        if not isinstance(val,dict) or not isinstance(val.get('name'),str) or not isinstance(val.get('arguments'),dict):
            return None
        calls.append(val)
    return calls

def evaluate_function_calls(labels,predictions):
    assert len(labels)==len(predictions),"label/prediction 개수가 다름"
    counts={k:{"correct":0,"total":0} for k in [
        "tool_decision_accuracy", "tool_selection", "params_selection",
        "params_value_accuracy", "exact_match_accuracy", "multi_call_exact_match"]}
    invalid_json=0; gold_tool_turns=0; predicted_extra_calls=0
    for label,pred in zip(labels,predictions):
        lc=extract_tool_calls(label);pc=extract_tool_calls(pred)
        if lc is None:raise ValueError('Gold label에 유효하지 않은 tool_call JSON이 존재')
        if pc is None:invalid_json+=1;pc=[]
        gold_tool=bool(lc);pred_tool=bool(pc)
        counts['tool_decision_accuracy']['total']+=1
        if gold_tool==pred_tool and (pred_tool or '<tool_call>' not in str(pred)):
            counts['tool_decision_accuracy']['correct']+=1
        if not gold_tool:
            if pred_tool:predicted_extra_calls+=1
            continue
        gold_tool_turns+=1
        counts['exact_match_accuracy']['total']+=1
        if len(lc)>1:counts['multi_call_exact_match']['total']+=1
        # Multi-call은 순서 무관한 개별 함수+인자 비교. 중복 호출도 그대로 비교
        def canon(c):return json.dumps({'name':c['name'],'arguments':c['arguments']},sort_keys=True,ensure_ascii=False)
        if sorted(map(canon,lc))==sorted(map(canon,pc)):
            counts['exact_match_accuracy']['correct']+=1
            if len(lc)>1:counts['multi_call_exact_match']['correct']+=1
        # 위치별 짝 비교: 함수명/파라미터 집계 분모는 실제 정답 호출 수에 맞춤
        for idx, expected in enumerate(lc):
            actual=pc[idx] if idx<len(pc) else None
            counts['tool_selection']['total']+=1
            if actual is not None and actual['name']==expected['name']:
                counts['tool_selection']['correct']+=1
            true_args=expected['arguments'];pred_args=actual['arguments'] if actual else {}
            true_keys=set(true_args);pred_keys=set(pred_args)
            counts['params_selection']['total']+=len(true_keys | pred_keys) or 1
            counts['params_selection']['correct']+=len(true_keys & pred_keys) if (true_keys | pred_keys) else int(actual is not None)
            counts['params_value_accuracy']['total']+=len(true_keys | pred_keys) or 1
            counts['params_value_accuracy']['correct']+=sum(true_args[k]==pred_args[k] for k in true_keys & pred_keys) if (true_keys | pred_keys) else int(actual is not None)
    final={key:(v['correct']/v['total'] if v['total'] else 0.0) for key,v in counts.items()}
    final.update({'total_samples':gold_tool_turns,'total_decision_turns':len(labels),
                  'invalid_tool_json':invalid_json,'extra_call_on_no_tool_turn':predicted_extra_calls})
    return final

labels=df['label'].to_list();preds=df['output'].to_list()
results_with_errors=evaluate_function_calls(labels,preds)
for metric,value in results_with_errors.items():
    print(f'{metric}: {value:.2%}' if metric.endswith('accuracy') or metric in ('tool_selection','params_selection') else f'{metric}: {value}')


tool_decision_accuracy: 91.54%
tool_selection: 85.00%
params_selection: 74.47%
params_value_accuracy: 73.40%
exact_match_accuracy: 66.67%
multi_call_exact_match: 0.75
total_samples: 48
total_decision_turns: 130
invalid_tool_json: 0
extra_call_on_no_tool_turn: 6


## **4.Base Model 호출**

In [8]:
import gc
import torch

# Base와 Fine-Tuned 모델을 동시에 GPU에 올리지 않도록 메모리 해제
try:
    del llm
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

base_llm = LLM(model="Qwen/Qwen2.5-7B-Instruct")
base_outputs = base_llm.generate(prompt_lst, sampling_params=sampling_params)
base_text_results = [sample.outputs[0].text.strip() for sample in base_outputs]
print(base_text_results[20])

df_base = pd.DataFrame({"prompt": prompt_lst, "label": label_lst, "output": base_text_results})
df_base.to_csv("evaluation_results_base.csv", index=False, encoding="utf-8-sig")
print("\n평가 결과가 'evaluation_results_base.csv' 파일로 저장되었습니다.")
df_base.head(10)

for label, pred in zip(df_base["label"].to_list()[:50], df_base["output"].to_list()[:50]):
    print("레이블 :", label)
    print("--" * 50)
    print("모델의 예측:", pred)
    print("==" * 50)

INFO 11-12 08:25:51 [utils.py:253] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
INFO 11-12 08:25:51 [model.py:631] Resolved architecture: Qwen2ForCausalLM
INFO 11-12 08:25:51 [model.py:1745] Using max model len 32768
INFO 11-12 08:25:51 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=8292) INFO 11-12 08:25:51 [core.py:93] Initializing a V1 LLM engine (v0.11.1) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='a

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(EngineCore_DP0 pid=8292) INFO 11-12 08:25:56 [default_loader.py:314] Loading weights took 1.83 seconds
(EngineCore_DP0 pid=8292) INFO 11-12 08:25:56 [gpu_model_runner.py:3334] Model loading took 14.2488 GiB memory and 2.360948 seconds
(EngineCore_DP0 pid=8292) INFO 11-12 08:26:02 [backends.py:631] Using cache directory: /root/.cache/vllm/torch_compile_cache/b79955c058/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=8292) INFO 11-12 08:26:02 [backends.py:647] Dynamo bytecode transform time: 4.94 s
(EngineCore_DP0 pid=8292) INFO 11-12 08:26:05 [backends.py:251] Cache the graph for dynamic shape for later use
(EngineCore_DP0 pid=8292) INFO 11-12 08:26:12 [backends.py:282] Compiling a graph for dynamic shape takes 9.57 s
(EngineCore_DP0 pid=8292) INFO 11-12 08:26:16 [monitor.py:34] torch.compile takes 14.51 s in total
(EngineCore_DP0 pid=8292) INFO 11-12 08:26:17 [gpu_worker.py:359] Available KV cache memory: 55.59 GiB
(EngineCore_DP0 pid=8292) INFO 11-12 08:26:17 [kv_cache

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 24.11it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 29.32it/s]


(EngineCore_DP0 pid=8292) INFO 11-12 08:26:21 [gpu_model_runner.py:4240] Graph capturing finished in 4 secs, took 0.49 GiB
(EngineCore_DP0 pid=8292) INFO 11-12 08:26:21 [core.py:250] init engine (profile, create kv cache, warmup model) took 25.01 seconds
INFO 11-12 08:26:23 [llm.py:352] Supported tasks: ['generate']


Adding requests:   0%|          | 0/130 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/130 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

월요일에는 14:00과 18:00에 각각 1:1 개인 레슨이 예약 가능합니다. 수요일에는 11:00, 16:00, 21:00에 1:1 개인 레슨이 예약 가능합니다. 원하시는 시간대를 선택하실 수 있습니다. 예약하시려면 언제에 예약을 진행하시겠습니까?

평가 결과가 'evaluation_results_base.csv' 파일로 저장되었습니다.
레이블 : 안녕하세요! 글로리 필라테스 AI 상담사입니다. 어떤 요일이나 수업 유형을 특정해서 조회해 드릴까요?
----------------------------------------------------------------------------------------------------
모델의 예측: 안녕하세요! 필라테스 수업 시간표를 확인해드리겠습니다. 월요일부터 금요일까지의 시간표를 알고 싶으신가요? 아니면 특정 요일만 알고 싶으신가요?
레이블 : <tool_call>
{"name": "get_class_schedule", "arguments": {"day_of_week": "월요일"}}
</tool_call>
<tool_call>
{"name": "get_class_schedule", "arguments": {"day_of_week": "수요일"}}
</tool_call>
----------------------------------------------------------------------------------------------------
모델의 예측: <tool_call>
{"name": "get_class_schedule", "arguments": {"day_of_week": "월요일, 수요일"}}
</tool_call>
레이블 : 월요일 수업:  
- 09:00~09:50 김민지 강사님의 7:1 레슨  
- 11:00~11:50 박지훈 강사님의 2:1 듀엣 레슨  
- 18:00~18:50 이서진 강사님의 1:1 개인 레슨  

수요일 수업:  
- 10:00~10:50 김민지 강사님의 7:1 레슨

In [9]:
base_labels = df_base["label"].to_list()
base_preds = df_base["output"].to_list()

base_results = evaluate_function_calls(base_labels, base_preds)
for metric, value in base_results.items():
    if metric != "total_samples":
        print(f"{metric}: {value:.2%}")
    else:
        print(f"{metric}: {value}")

tool_decision_accuracy: 85.38%
tool_selection: 56.67%
params_selection: 51.55%
params_value_accuracy: 47.42%
exact_match_accuracy: 31.25%
multi_call_exact_match: 8.33%
total_samples: 48
total_decision_turns: 13000.00%
invalid_tool_json: 0.00%
extra_call_on_no_tool_turn: 600.00%
